# 훈련 추천 지표 준비

일별 훈련 데이터를 이용하여 오늘의 훈련 추천에 사용할 후보 지표를 만든다.

## 1. 추천 시점과 입력 정보

오늘 아침의 훈련을 추천한다고 가정한다. 추천 시점에는 오늘의 운동 결과를 알 수 없으므로, 추천일 이전까지 기록된 데이터만 사용한다.

추천 지표 후보는 다음과 같다.

- 직전 7일 누적 TSS
- 직전 28일 누적 TSS
- 단기 부하와 장기 부하의 관계
- 전날까지의 연속 라이드 일수
- 최근 무기록 일수
- 직전 7일 운동 시간과 라이드 횟수

TSS는 기록된 훈련 부하를 요약한 값이며, 실제 피로나 회복 상태를 직접 측정하는 값은 아니다. 또한 라이드 무기록일을 실제 휴식일로 단정하지 않는다.

In [26]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()
processed_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_bike_rides_cleaned.csv"
)

rides_df = pd.read_csv(
    processed_path,
    parse_dates=["date"],
)

analysis_period_df = rides_df.loc[
    (rides_df["date"] >= "2009-02-01")
    & (rides_df["date"] < "2009-07-01")
].copy()

analysis_period_df.shape

(111, 63)

In [27]:
daily_training_df = (
    analysis_period_df
    .set_index("date")
    .resample("D")
    .agg(
        workout_hours=("workout_hours", "sum"),
        tss=("coggan_tss", "sum"),
        ride_count=("sport", "size"),
    )
)

daily_training_df.head(10)

,workout_hours,tss,ride_count
date,,,
2009-02-07 00:00:00+00:00,1.193333,111.22867,1
2009-02-08 00:00:00+00:00,2.460833,176.54760,1
2009-02-09 00:00:00+00:00,0.000000,0.00000,0
2009-02-10 00:00:00+00:00,0.833333,43.90424,1
2009-02-11 00:00:00+00:00,1.000000,50.11669,1
2009-02-12 00:00:00+00:00,2.000000,100.23338,2
2009-02-13 00:00:00+00:00,1.124167,108.21922,1
2009-02-14 00:00:00+00:00,1.970278,116.20702,3
2009-02-15 00:00:00+00:00,1.962222,186.06192,1


In [28]:
daily_training_df["has_recorded_ride"] = (
    daily_training_df["ride_count"] > 0
)

daily_training_df["has_recorded_ride"].value_counts()

has_recorded_ride
True     93
False    49
Name: count, dtype: int64

In [29]:
daily_training_df["previous_7_day_tss"] = (
    daily_training_df["tss"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["tss", "previous_7_day_tss"]
].head(10).round(2)

,tss,previous_7_day_tss
date,,
2009-02-07 00:00:00+00:00,111.23,NaN
2009-02-08 00:00:00+00:00,176.55,NaN
2009-02-09 00:00:00+00:00,0.00,NaN
2009-02-10 00:00:00+00:00,43.90,NaN
2009-02-11 00:00:00+00:00,50.12,NaN
2009-02-12 00:00:00+00:00,100.23,NaN
2009-02-13 00:00:00+00:00,108.22,NaN
2009-02-14 00:00:00+00:00,116.21,590.25
2009-02-15 00:00:00+00:00,186.06,595.23


In [30]:
daily_training_df["previous_28_day_tss"] = (
    daily_training_df["tss"]
    .shift(1)
    .rolling(window=28, min_periods=28)
    .sum()
)

daily_training_df[
    ["previous_7_day_tss", "previous_28_day_tss"]
].loc["2009-03-05":"2009-03-09"].round(2)

,previous_7_day_tss,previous_28_day_tss
date,,
2009-03-05 00:00:00+00:00,776.14,NaN
2009-03-06 00:00:00+00:00,676.18,NaN
2009-03-07 00:00:00+00:00,531.81,2494.51
2009-03-08 00:00:00+00:00,582.35,2511.41
2009-03-09 00:00:00+00:00,348.77,2334.86


In [31]:
daily_training_df["previous_28_day_tss_weekly_average"] = (
    daily_training_df["previous_28_day_tss"] / 4
)

daily_training_df["short_to_long_tss_ratio"] = (
    daily_training_df["previous_7_day_tss"]
    / daily_training_df["previous_28_day_tss_weekly_average"]
)

daily_training_df[
    [
        "previous_7_day_tss",
        "previous_28_day_tss_weekly_average",
        "short_to_long_tss_ratio",
    ]
].loc["2009-03-07":"2009-03-11"].round(2)

,previous_7_day_tss,previous_28_day_tss_weekly_average,short_to_long_tss_ratio
date,,,
2009-03-07 00:00:00+00:00,531.81,623.63,0.85
2009-03-08 00:00:00+00:00,582.35,627.85,0.93
2009-03-09 00:00:00+00:00,348.77,583.72,0.60
2009-03-10 00:00:00+00:00,348.92,612.11,0.57
2009-03-11 00:00:00+00:00,348.92,601.14,0.58


In [32]:
daily_training_df["previous_7_day_workout_hours"] = (
    daily_training_df["workout_hours"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["workout_hours", "previous_7_day_workout_hours"]
].head(10).round(2)

,workout_hours,previous_7_day_workout_hours
date,,
2009-02-07 00:00:00+00:00,1.19,NaN
2009-02-08 00:00:00+00:00,2.46,NaN
2009-02-09 00:00:00+00:00,0.00,NaN
2009-02-10 00:00:00+00:00,0.83,NaN
2009-02-11 00:00:00+00:00,1.00,NaN
2009-02-12 00:00:00+00:00,2.00,NaN
2009-02-13 00:00:00+00:00,1.12,NaN
2009-02-14 00:00:00+00:00,1.97,8.61
2009-02-15 00:00:00+00:00,1.96,9.39


In [33]:
daily_training_df["previous_7_day_ride_count"] = (
    daily_training_df["ride_count"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["ride_count", "previous_7_day_ride_count"]
].head(10)

,ride_count,previous_7_day_ride_count
date,,
2009-02-07 00:00:00+00:00,1,NaN
2009-02-08 00:00:00+00:00,1,NaN
2009-02-09 00:00:00+00:00,0,NaN
2009-02-10 00:00:00+00:00,1,NaN
2009-02-11 00:00:00+00:00,1,NaN
2009-02-12 00:00:00+00:00,2,NaN
2009-02-13 00:00:00+00:00,1,NaN
2009-02-14 00:00:00+00:00,3,7.0
2009-02-15 00:00:00+00:00,1,9.0


In [34]:
daily_training_df["ride_status_changed"] = (
    daily_training_df["has_recorded_ride"]
    .ne(daily_training_df["has_recorded_ride"].shift())
)

daily_training_df["ride_status_streak_id"] = (
    daily_training_df["ride_status_changed"]
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "ride_status_changed",
        "ride_status_streak_id",
    ]
].head(10)

,has_recorded_ride,ride_status_changed,ride_status_streak_id
date,,,
2009-02-07 00:00:00+00:00,True,True,1
2009-02-08 00:00:00+00:00,True,False,1
2009-02-09 00:00:00+00:00,False,True,2
2009-02-10 00:00:00+00:00,True,True,3
2009-02-11 00:00:00+00:00,True,False,3
2009-02-12 00:00:00+00:00,True,False,3
2009-02-13 00:00:00+00:00,True,False,3
2009-02-14 00:00:00+00:00,True,False,3
2009-02-15 00:00:00+00:00,True,False,3


In [35]:
daily_training_df["recorded_ride_streak_ending_today"] = (
    daily_training_df["has_recorded_ride"]
    .groupby(daily_training_df["ride_status_streak_id"])
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "ride_status_streak_id",
        "recorded_ride_streak_ending_today",
    ]
].head(10)

,has_recorded_ride,ride_status_streak_id,recorded_ride_streak_ending_today
date,,,
2009-02-07 00:00:00+00:00,True,1,1
2009-02-08 00:00:00+00:00,True,1,2
2009-02-09 00:00:00+00:00,False,2,0
2009-02-10 00:00:00+00:00,True,3,1
2009-02-11 00:00:00+00:00,True,3,2
2009-02-12 00:00:00+00:00,True,3,3
2009-02-13 00:00:00+00:00,True,3,4
2009-02-14 00:00:00+00:00,True,3,5
2009-02-15 00:00:00+00:00,True,3,6


In [36]:
daily_training_df["previous_day_recorded_ride_streak_days"] = (
    daily_training_df["recorded_ride_streak_ending_today"]
    .shift(1)
)

daily_training_df[
    [
        "recorded_ride_streak_ending_today",
        "previous_day_recorded_ride_streak_days",
    ]
].head(10)

,recorded_ride_streak_ending_today,previous_day_recorded_ride_streak_days
date,,
2009-02-07 00:00:00+00:00,1,NaN
2009-02-08 00:00:00+00:00,2,1.0
2009-02-09 00:00:00+00:00,0,2.0
2009-02-10 00:00:00+00:00,1,0.0
2009-02-11 00:00:00+00:00,2,1.0
2009-02-12 00:00:00+00:00,3,2.0
2009-02-13 00:00:00+00:00,4,3.0
2009-02-14 00:00:00+00:00,5,4.0
2009-02-15 00:00:00+00:00,6,5.0


In [37]:
daily_training_df["no_record_streak_ending_today"] = (
    (~daily_training_df["has_recorded_ride"])
    .groupby(daily_training_df["ride_status_streak_id"])
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "no_record_streak_ending_today",
    ]
].head(10)

,has_recorded_ride,no_record_streak_ending_today
date,,
2009-02-07 00:00:00+00:00,True,0
2009-02-08 00:00:00+00:00,True,0
2009-02-09 00:00:00+00:00,False,1
2009-02-10 00:00:00+00:00,True,0
2009-02-11 00:00:00+00:00,True,0
2009-02-12 00:00:00+00:00,True,0
2009-02-13 00:00:00+00:00,True,0
2009-02-14 00:00:00+00:00,True,0
2009-02-15 00:00:00+00:00,True,0


In [38]:
daily_training_df["previous_day_no_record_streak_days"] = (
    daily_training_df["no_record_streak_ending_today"]
    .shift(1)
)

daily_training_df[
    [
        "no_record_streak_ending_today",
        "previous_day_no_record_streak_days",
    ]
].head(10)

,no_record_streak_ending_today,previous_day_no_record_streak_days
date,,
2009-02-07 00:00:00+00:00,0,NaN
2009-02-08 00:00:00+00:00,0,0.0
2009-02-09 00:00:00+00:00,1,0.0
2009-02-10 00:00:00+00:00,0,1.0
2009-02-11 00:00:00+00:00,0,0.0
2009-02-12 00:00:00+00:00,0,0.0
2009-02-13 00:00:00+00:00,0,0.0
2009-02-14 00:00:00+00:00,0,0.0
2009-02-15 00:00:00+00:00,0,0.0


In [39]:
recommendation_input_columns = [
    "previous_7_day_tss",
    "previous_28_day_tss",
    "short_to_long_tss_ratio",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

recommendation_inputs_df = daily_training_df[
    recommendation_input_columns
].copy()

recommendation_inputs_df.loc[
    "2009-03-07":"2009-03-11"
].round(2)

,previous_7_day_tss,previous_28_day_tss,short_to_long_tss_ratio,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
date,,,,,,,
2009-03-07 00:00:00+00:00,531.81,2494.51,0.85,7.54,4.0,1.0,0.0
2009-03-08 00:00:00+00:00,582.35,2511.41,0.93,7.57,4.0,2.0,0.0
2009-03-09 00:00:00+00:00,348.77,2334.86,0.60,4.37,3.0,0.0,1.0
2009-03-10 00:00:00+00:00,348.92,2448.45,0.57,3.81,3.0,1.0,0.0
2009-03-11 00:00:00+00:00,348.92,2404.54,0.58,3.81,3.0,0.0,1.0


In [40]:
manual_previous_7_day_tss = (
    daily_training_df
    .loc["2009-03-03":"2009-03-09", "tss"]
    .sum()
)

stored_previous_7_day_tss = (
    recommendation_inputs_df
    .loc["2009-03-10", "previous_7_day_tss"]
)

pd.Series(
    {
        "manual_previous_7_day_tss": manual_previous_7_day_tss,
        "stored_previous_7_day_tss": stored_previous_7_day_tss,
        "difference": (
            manual_previous_7_day_tss
            - stored_previous_7_day_tss
        ),
    }
).round(2)

manual_previous_7_day_tss    348.92
stored_previous_7_day_tss    348.92
difference                     0.00
dtype: float64

In [41]:
manual_previous_28_day_tss = (
    daily_training_df
    .loc["2009-02-10":"2009-03-09", "tss"]
    .sum()
)

stored_previous_28_day_tss = (
    recommendation_inputs_df
    .loc["2009-03-10", "previous_28_day_tss"]
)

pd.Series(
    {
        "manual_previous_28_day_tss": manual_previous_28_day_tss,
        "stored_previous_28_day_tss": stored_previous_28_day_tss,
        "difference": (
            manual_previous_28_day_tss
            - stored_previous_28_day_tss
        ),
    }
).round(2)

manual_previous_28_day_tss    2448.45
stored_previous_28_day_tss    2448.45
difference                       0.00
dtype: float64

In [42]:
recommendation_ready_df = (
    recommendation_inputs_df
    .dropna()
    .copy()
)

pd.Series(
    {
        "row_count": len(recommendation_ready_df),
        "first_date": recommendation_ready_df.index.min(),
        "last_date": recommendation_ready_df.index.max(),
    }
)

row_count                           114
first_date    2009-03-07 00:00:00+00:00
last_date     2009-06-28 00:00:00+00:00
dtype: object

In [44]:
recommendation_ready_df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
previous_7_day_tss,114.0,679.85,158.70,348.77,580.91,704.64,796.78,1012.40
previous_28_day_tss,114.0,2751.09,287.28,2213.47,2483.50,2810.09,3003.21,3309.16
short_to_long_tss_ratio,114.0,0.99,0.20,0.55,0.85,0.98,1.12,1.39
previous_7_day_workout_hours,114.0,9.45,2.54,3.81,7.77,9.62,11.29,14.86
previous_7_day_ride_count,114.0,5.31,1.47,3.00,4.00,5.00,6.00,8.00
previous_day_recorded_ride_streak_days,114.0,1.96,2.41,0.00,0.00,1.00,3.00,12.00
previous_day_no_record_streak_days,114.0,0.54,0.87,0.00,0.00,0.00,1.00,4.00


### 추천 입력값 분포 확인 결과

- 직전 28일 이력이 확보된 2009년 3월 7일부터 6월 28일까지 총 114일의 추천 입력값을 확인했다.
- 직전 7일 TSS의 중앙값은 약 704.64였고, 25% 지점부터 75% 지점까지의 범위는 약 580.91에서 796.78이었다.
- 단기/장기 TSS 비율의 중앙값은 0.98, 평균은 0.99였다. 이 기간에서 최근 7일의 기록된 TSS는 직전 28일의 주당 평균과 대체로 비슷한 흐름을 보였다.
- 단기/장기 TSS 비율의 25%와 75% 지점은 각각 0.85와 1.12였다.
- 직전 7일 운동 시간의 중앙값은 약 9.62시간이고, 라이드 횟수의 중앙값은 5회였다.
- 전날까지의 연속 라이드 기록일 수 중앙값은 1일이며, 전날까지의 연속 무기록일 수 중앙값은 0일이었다.
- 이후 규칙을 설계할 때는 추천일 이전 데이터만 사용해 기준값을 계산한다.

TSS와 단기/장기 TSS 비율은 기록된 훈련 부하 흐름을 요약한 값이며, 실제 피로·회복 상태를 직접 측정하지 않는다. 또한 라이드 무기록일은 실제 휴식일로 단정하지 않는다.

In [47]:
daily_training_df["previous_ratio_q25"] = (
    daily_training_df["short_to_long_tss_ratio"]
    .shift(1)
    .expanding(min_periods=14)
    .quantile(0.25)
)

daily_training_df["previous_ratio_q75"] = (
    daily_training_df["short_to_long_tss_ratio"]
    .shift(1)
    .expanding(min_periods=14)
    .quantile(0.75)
)

daily_training_df[
    [
        "short_to_long_tss_ratio",
        "previous_ratio_q25",
        "previous_ratio_q75",
    ]
].loc["2009-03-19":"2009-03-23"].round(2)

,short_to_long_tss_ratio,previous_ratio_q25,previous_ratio_q75
date,,,
2009-03-19 00:00:00+00:00,1.15,NaN,NaN
2009-03-20 00:00:00+00:00,1.15,NaN,NaN
2009-03-21 00:00:00+00:00,1.32,0.60,1.01
2009-03-22 00:00:00+00:00,1.39,0.61,1.08
2009-03-23 00:00:00+00:00,1.34,0.61,1.15


In [48]:
has_ratio_reference = (
    daily_training_df["previous_ratio_q25"].notna()
    & daily_training_df["previous_ratio_q75"].notna()
)

daily_training_df["relative_recent_load_level"] = pd.NA

daily_training_df.loc[
    has_ratio_reference
    & (
        daily_training_df["short_to_long_tss_ratio"]
        < daily_training_df["previous_ratio_q25"]
    ),
    "relative_recent_load_level",
] = "relatively_low"

daily_training_df.loc[
    has_ratio_reference
    & (
        daily_training_df["short_to_long_tss_ratio"]
        > daily_training_df["previous_ratio_q75"]
    ),
    "relative_recent_load_level",
] = "relatively_high"

daily_training_df.loc[
    has_ratio_reference
    & daily_training_df["relative_recent_load_level"].isna(),
    "relative_recent_load_level",
] = "typical_range"

daily_training_df[
    [
        "short_to_long_tss_ratio",
        "previous_ratio_q25",
        "previous_ratio_q75",
        "relative_recent_load_level",
    ]
].loc["2009-03-19":"2009-03-23"].round(2)

,short_to_long_tss_ratio,previous_ratio_q25,previous_ratio_q75,relative_recent_load_level
date,,,,
2009-03-19 00:00:00+00:00,1.15,NaN,NaN,<NA>
2009-03-20 00:00:00+00:00,1.15,NaN,NaN,<NA>
2009-03-21 00:00:00+00:00,1.32,0.60,1.01,relatively_high
2009-03-22 00:00:00+00:00,1.39,0.61,1.08,relatively_high
2009-03-23 00:00:00+00:00,1.34,0.61,1.15,relatively_high


In [49]:
daily_training_df[
    "relative_recent_load_level"
].value_counts(dropna=False)

relative_recent_load_level
typical_range      58
<NA>               42
relatively_high    23
relatively_low     19
Name: count, dtype: int64

In [50]:
daily_training_df.groupby(
    "relative_recent_load_level"
)["short_to_long_tss_ratio"].agg(
    ["count", "min", "median", "max"]
).round(2)

,count,min,median,max
relative_recent_load_level,,,,
relatively_high,23,1.15,1.24,1.39
relatively_low,19,0.55,0.76,0.89
typical_range,58,0.78,0.98,1.15


In [51]:
context_columns = [
    "relative_recent_load_level",
    "short_to_long_tss_ratio",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

daily_training_df.loc[
    daily_training_df["relative_recent_load_level"]
    == "relatively_high",
    context_columns,
].head(5).round(2)

,relative_recent_load_level,short_to_long_tss_ratio,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
date,,,,,,
2009-03-21 00:00:00+00:00,relatively_high,1.32,11.19,8.0,1.0,0.0
2009-03-22 00:00:00+00:00,relatively_high,1.39,12.62,8.0,2.0,0.0
2009-03-23 00:00:00+00:00,relatively_high,1.34,10.73,6.0,3.0,0.0
2009-03-24 00:00:00+00:00,relatively_high,1.17,9.16,5.0,0.0,1.0
2009-04-03 00:00:00+00:00,relatively_high,1.24,8.98,5.0,1.0,0.0


In [53]:
context_columns = [
    "relative_recent_load_level",
    "short_to_long_tss_ratio",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

daily_training_df.loc[
    daily_training_df["relative_recent_load_level"]
    == "relatively_low",
    context_columns,
].head(5).round(2)

,relative_recent_load_level,short_to_long_tss_ratio,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
date,,,,,,
2009-03-29 00:00:00+00:00,relatively_low,0.68,5.16,3.0,2.0,0.0
2009-04-13 00:00:00+00:00,relatively_low,0.67,7.22,4.0,2.0,0.0
2009-04-15 00:00:00+00:00,relatively_low,0.69,6.15,4.0,4.0,0.0
2009-05-09 00:00:00+00:00,relatively_low,0.77,7.98,4.0,0.0,2.0
2009-05-12 00:00:00+00:00,relatively_low,0.75,8.72,3.0,0.0,1.0


In [54]:
daily_training_df.groupby(
    "relative_recent_load_level"
)[
    [
        "previous_7_day_workout_hours",
        "previous_7_day_ride_count",
        "previous_day_recorded_ride_streak_days",
        "previous_day_no_record_streak_days",
    ]
].median().round(2)

,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
relative_recent_load_level,,,,
relatively_high,12.29,6.0,2.0,0.0
relatively_low,7.97,4.0,1.0,0.0
typical_range,9.86,5.0,1.0,0.0


### 동적 부하 라벨과 훈련 맥락 확인 결과

- 각 추천일의 단기/장기 TSS 비율을 그 전날까지의 비율 분포와 비교해 상대적 부하 라벨을 생성했다.
- 과거 비율이 14개 이상 확보된 100일에 `relatively_low`, `typical_range`, `relatively_high` 라벨을 만들었다. 기준이 부족한 42일은 라벨을 생성하지 않았다.
- 중간 범위는 58일, 상대적으로 높은 부하는 23일, 상대적으로 낮은 부하는 19일이었다.
- 단기/장기 TSS 비율의 중앙값은 상대적으로 낮은 부하 0.76, 중간 범위 0.98, 상대적으로 높은 부하 1.24 순으로 나타났다.
- 직전 7일 운동 시간 중앙값은 상대적으로 높은 부하 그룹에서 12.29시간, 중간 범위에서 9.86시간, 상대적으로 낮은 부하 그룹에서 7.97시간이었다. 라이드 횟수 중앙값도 각각 6회, 5회, 4회였다.
- 전날까지의 연속 라이드 기록일 수 중앙값은 상대적으로 높은 부하 그룹에서 2일, 나머지 두 그룹에서 1일이었다. 전날까지의 연속 무기록일 수 중앙값은 세 그룹 모두 0일로 나타났다.

이 라벨은 각 추천일 이전에 기록된 TSS 흐름과 비교한 상대적 위치를 나타낸다. 실제 피로·회복 상태를 직접 측정하지 않으며, 특정 훈련 수행 여부를 결정하는 기준으로 사용하지 않는다. 라이드 무기록일도 실제 휴식일로 단정하지 않는다.